# Part 4 - Reporting Tool

Edit the configuration values and run the notebook to generate a report from the SQLite database.

In [1]:

from pathlib import Path
import sqlite3
from datetime import datetime, timedelta

# ==========================
# Configuration
# ==========================
REPORT_TYPE = "monthly"      # daily, weekly, monthly
START_DATE = "2025-01-01"    # YYYY-MM-DD
END_DATE   = "2025-01-31"    # YYYY-MM-DD

DB_PATH = Path("../../Part 3/database/ecommerce.db")
REVENUE_EXPR = "oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)"

def parse_date(s):
    return datetime.strptime(s, "%Y-%m-%d").date()

def previous_period(start_date, end_date):
    days = (end_date - start_date).days + 1
    prev_end = start_date - timedelta(days=1)
    prev_start = prev_end - timedelta(days=days-1)
    return prev_start, prev_end

def get_period_metrics(conn, start_date, end_date):
    cur = conn.cursor()

    cur.execute(f"""
        SELECT
            COUNT(DISTINCT o.order_id),
            COALESCE(SUM({REVENUE_EXPR}),0),
            COUNT(DISTINCT o.customer_id)
        FROM orders o
        LEFT JOIN order_items oi ON oi.order_id=o.order_id
        WHERE date(o.order_date) BETWEEN ? AND ?
    """,(start_date.isoformat(), end_date.isoformat()))
    total_orders,total_revenue,unique_customers = cur.fetchone()

    cur.execute(f"""
        SELECT p.product_name,SUM({REVENUE_EXPR}) revenue
        FROM order_items oi
        JOIN orders o ON oi.order_id=o.order_id
        JOIN products p ON oi.product_id=p.product_id
        WHERE date(o.order_date) BETWEEN ? AND ?
        GROUP BY p.product_id,p.product_name
        ORDER BY revenue DESC
        LIMIT 3
    """,(start_date.isoformat(), end_date.isoformat()))
    top_products = cur.fetchall()

    return {
        "orders": total_orders or 0,
        "revenue": round(total_revenue or 0,2),
        "customers": unique_customers or 0,
        "top_products": top_products
    }

def pct_change(current, previous):
    if previous == 0:
        return None
    return round((current-previous)*100/previous,2)

def fmt(label,current,previous,currency=False):
    ch = pct_change(current,previous)
    prev = f"Rs. {previous:,.2f}" if currency else previous
    if ch is None:
        return f"{label:<18}: N/A (previous={prev})"
    direction = "▲" if ch>=0 else "▼"
    return f"{label:<18}: {current} ({direction} {ch:+.2f}% vs previous {prev})"

start = parse_date(START_DATE)
end = parse_date(END_DATE)

conn = sqlite3.connect(DB_PATH)

current = get_period_metrics(conn,start,end)
prev_start,prev_end = previous_period(start,end)
previous = get_period_metrics(conn,prev_start,prev_end)

print("="*65)
print(f"{REPORT_TYPE.upper()} SALES REPORT")
print("="*65)
print(f"Period : {start} to {end}\n")

print(f"Total Orders      : {current['orders']}")
print(f"Total Revenue     : Rs. {current['revenue']:,.2f}")
print(f"Unique Customers  : {current['customers']}\n")

print("Top 3 Products")
for i,(name,rev) in enumerate(current["top_products"],1):
    print(f"{i}. {name:<35} Rs. {rev:,.2f}")

print("\nComparison with Previous Period")
print(f"({prev_start} to {prev_end})")
print(fmt("Orders",current["orders"],previous["orders"]))
print(fmt("Revenue",current["revenue"],previous["revenue"],True))
print(fmt("Customers",current["customers"],previous["customers"]))

conn.close()


MONTHLY SALES REPORT
Period : 2025-01-01 to 2025-01-31

Total Orders      : 90
Total Revenue     : Rs. 3,130,238.42
Unique Customers  : 86

Top 3 Products
1. Lumen Phone Elite                   Rs. 257,368.24
2. Aero Phone Signature                Rs. 233,967.20
3. Prime Phone Basic                   Rs. 215,300.52

Comparison with Previous Period
(2024-12-01 to 2024-12-31)
Orders            : 90 (▼ -20.35% vs previous 113)
Revenue           : 3130238.42 (▼ -32.89% vs previous Rs. 4,664,107.52)
Customers         : 86 (▼ -12.24% vs previous 98)
